# 🏦 Pipeline de Categorización de Transacciones Bancarias
### Clasificador local (nivel 1) + fallback LLM (nivel 2) + anonimización + anomalías

**Estructura del notebook:**
1. Setup y dependencias
2. Dataset — carga, normalización, sintético, split 80/10/10
3. Embeddings y entrenamiento del clasificador (LogReg / LinearSVC / LightGBM)
4. Calibración de probabilidades y umbral de confianza
5. Fallback LLM (nivel 2) — prompt con few-shot
6. Anonimización pre-LLM (Presidio + regex)
7. Detección de anomalías (Z-score + suscripciones)
8. Insights mensuales (batch → LLM)
9. Evaluación final — métricas, matriz de confusión, análisis de errores
10. Serialización de artefactos `/models` + `classify()` lista para endpoint


## 0. Setup y dependencias

In [ ]:
# Instalar dependencias (solo la primera vez)
import subprocess, sys

pkgs = [
    "sentence-transformers",
    "scikit-learn",
    "lightgbm",
    "joblib",
    "presidio-analyzer",
    "presidio-anonymizer",
    "spacy",
    "imbalanced-learn",
    "matplotlib",
    "seaborn",
]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "--break-system-packages", "-q"],
                   capture_output=True)
print("Dependencias instaladas ✓")


In [ ]:
import json, re, time, warnings, random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 110
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Imports OK ✓")
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


## 1. Taxonomía de categorías

In [ ]:
# ── Taxonomía cerrada: 12 categorías canónicas ──────────────────────────────
CATEGORIES = [
    "alimentacion", "compras", "hogar", "impuestos_tasas",
    "ingresos", "ocio", "otros", "restauracion",
    "salud", "suscripciones", "transporte", "transferencias",
]

CATEGORY_ALIASES = {
    " ingresos": "ingresos",
    "transportista": "transporte",
    "transferencia": "transferencias",
    "alcohol": "ocio",
    "decoracion ": "hogar",
    "decoracion": "hogar",
    "viajes": "ocio",
    "food": "restauracion",
}

def normalize_category(cat):
    if cat is None:
        return "otros"
    cat = str(cat).strip().lower()
    if cat in CATEGORY_ALIASES:
        return CATEGORY_ALIASES[cat]
    return cat if cat in CATEGORIES else "otros"

print(f"Categorías ({len(CATEGORIES)}):", CATEGORIES)


## 2. Dataset

### 2.1 Carga del JSONL limpio

In [ ]:
JSONL_PATH = "/mnt/user-data/outputs/transactions_clean.jsonl"   # ← ajusta si es necesario

def load_clean_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            obj = json.loads(line.strip())
            cat = normalize_category(obj.get("categoria"))
            if cat not in CATEGORIES:
                continue
            desc = str(obj.get("descripcion") or "").strip()
            imp  = obj.get("importe")
            if not desc or imp is None:
                continue
            try:
                imp = float(imp)
            except (TypeError, ValueError):
                continue
            records.append({
                "descripcion": desc,
                "importe": imp,
                "categoria": cat,
                "comercio_normalizado": obj.get("comercio_normalizado"),
                "source": "real",
            })
    return pd.DataFrame(records)

df_real = load_clean_jsonl(JSONL_PATH)
print(f"Registros reales: {len(df_real)}")
print("\nDistribución por categoría:")
print(df_real["categoria"].value_counts().to_string())


### 2.2 Validación del muestreo manual (5-10%)

In [ ]:
# Muestra manual para validar etiquetas (muestreo estratificado 5%)
VALIDATION_SAMPLE_PCT = 0.05

sample = (
    df_real.groupby("categoria", group_keys=False)
    .apply(lambda g: g.sample(max(1, int(len(g) * VALIDATION_SAMPLE_PCT)), random_state=SEED))
    .reset_index(drop=True)
)
print(f"Muestra de validación manual: {len(sample)} registros ({VALIDATION_SAMPLE_PCT*100:.0f}%)")
print("\nMuestra (primeras 10 filas):")
print(sample[["descripcion", "importe", "categoria"]].head(10).to_string(index=False))

# Guardar la muestra para revisión manual
sample.to_csv("/tmp/validacion_manual.csv", index=False)
print("\n💾 Guardada en /tmp/validacion_manual.csv para revisión manual")


### 2.3 Generación de datos sintéticos

In [ ]:
# Plantillas sintéticas por categoría para balancear el dataset
SYNTHETIC_TEMPLATES = {
    "alimentacion": [
        ("MERCADONA {n}", lambda: -round(random.uniform(15, 120), 2)),
        ("LIDL COMPRA {n}", lambda: -round(random.uniform(10, 80), 2)),
        ("SUPERMERCADO DIA {n}", lambda: -round(random.uniform(8, 60), 2)),
        ("CARREFOUR ONLINE {n}", lambda: -round(random.uniform(30, 150), 2)),
        ("EROSKI {n}", lambda: -round(random.uniform(12, 90), 2)),
        ("ALCAMPO {n}", lambda: -round(random.uniform(20, 100), 2)),
        ("EL CORTE INGLES ALIMENTACION {n}", lambda: -round(random.uniform(15, 80), 2)),
    ],
    "compras": [
        ("AMAZON.ES {n}", lambda: -round(random.uniform(10, 300), 2)),
        ("ZARA {n}", lambda: -round(random.uniform(20, 150), 2)),
        ("FNAC {n}", lambda: -round(random.uniform(15, 400), 2)),
        ("EL CORTE INGLES {n}", lambda: -round(random.uniform(20, 500), 2)),
        ("PRIMARK {n}", lambda: -round(random.uniform(15, 120), 2)),
        ("MANGO {n}", lambda: -round(random.uniform(20, 180), 2)),
        ("ALIEXPRESS {n}", lambda: -round(random.uniform(3, 50), 2)),
    ],
    "hogar": [
        ("IKEA {n}", lambda: -round(random.uniform(20, 600), 2)),
        ("LEROY MERLIN {n}", lambda: -round(random.uniform(15, 400), 2)),
        ("COMUNIDAD PROPIETARIOS {n}", lambda: -round(random.uniform(50, 200), 2)),
        ("ENDESA FACTURA {n}", lambda: -round(random.uniform(40, 180), 2)),
        ("IBERDROLA LUZ {n}", lambda: -round(random.uniform(35, 160), 2)),
        ("GAS NATURAL {n}", lambda: -round(random.uniform(30, 120), 2)),
        ("ALQUILER PISO {n}", lambda: -round(random.uniform(400, 1500), 2)),
        ("HIPOTECA BBVA {n}", lambda: -round(random.uniform(400, 1200), 2)),
    ],
    "impuestos_tasas": [
        ("AGENCIA TRIBUTARIA {n}", lambda: -round(random.uniform(100, 2000), 2)),
        ("AYUNTAMIENTO MADRID IBI {n}", lambda: -round(random.uniform(200, 800), 2)),
        ("TASA BASURAS {n}", lambda: -round(random.uniform(30, 120), 2)),
        ("MULTA DGT {n}", lambda: -round(random.uniform(100, 600), 2)),
        ("SEGURIDAD SOCIAL {n}", lambda: -round(random.uniform(50, 300), 2)),
        ("ITPAJD IMPUESTO {n}", lambda: -round(random.uniform(200, 5000), 2)),
    ],
    "ingresos": [
        ("NOMINA EMPRESA {n}", lambda: round(random.uniform(1000, 4000), 2)),
        ("TRANSFERENCIA RECIBIDA {n}", lambda: round(random.uniform(50, 2000), 2)),
        ("DEVOLUCION HACIENDA {n}", lambda: round(random.uniform(100, 1500), 2)),
        ("INGRESO BIZUM {n}", lambda: round(random.uniform(5, 200), 2)),
        ("PENSION INGRESO {n}", lambda: round(random.uniform(600, 1800), 2)),
        ("PAGO FREELANCE {n}", lambda: round(random.uniform(200, 3000), 2)),
    ],
    "ocio": [
        ("CINES ODEON {n}", lambda: -round(random.uniform(8, 20), 2)),
        ("STEAM GAME {n}", lambda: -round(random.uniform(5, 70), 2)),
        ("ENTRADAS CONCIERTO {n}", lambda: -round(random.uniform(30, 150), 2)),
        ("ESTADIO BERNABEU {n}", lambda: -round(random.uniform(20, 200), 2)),
        ("BOWLING {n}", lambda: -round(random.uniform(8, 25), 2)),
        ("ESCAPE ROOM {n}", lambda: -round(random.uniform(15, 40), 2)),
        ("PAINTBALL {n}", lambda: -round(random.uniform(20, 50), 2)),
        ("PARQUE TEMATICO {n}", lambda: -round(random.uniform(30, 80), 2)),
    ],
    "restauracion": [
        ("MCDONALDS {n}", lambda: -round(random.uniform(5, 25), 2)),
        ("BURGER KING {n}", lambda: -round(random.uniform(5, 22), 2)),
        ("RESTAURANTE LA TABERNA {n}", lambda: -round(random.uniform(15, 80), 2)),
        ("GLOVO PEDIDO {n}", lambda: -round(random.uniform(12, 45), 2)),
        ("JUST EAT {n}", lambda: -round(random.uniform(10, 40), 2)),
        ("BAR CAFETERIA {n}", lambda: -round(random.uniform(2, 20), 2)),
        ("TELEPIZZA {n}", lambda: -round(random.uniform(8, 35), 2)),
        ("UBER EATS {n}", lambda: -round(random.uniform(10, 40), 2)),
    ],
    "salud": [
        ("FARMACIA AHORRO {n}", lambda: -round(random.uniform(5, 80), 2)),
        ("CLINICA DENTAL {n}", lambda: -round(random.uniform(30, 500), 2)),
        ("SANITAS SEGURO {n}", lambda: -round(random.uniform(40, 150), 2)),
        ("OPTICA 2000 {n}", lambda: -round(random.uniform(20, 300), 2)),
        ("HOSPITAL QUIRON {n}", lambda: -round(random.uniform(50, 800), 2)),
        ("FISIOTERAPIA {n}", lambda: -round(random.uniform(30, 80), 2)),
        ("FARMACIA GUARDIA {n}", lambda: -round(random.uniform(5, 60), 2)),
        ("GIMNASIO BASIC FIT {n}", lambda: -round(random.uniform(15, 40), 2)),
    ],
    "suscripciones": [
        ("NETFLIX {n}", lambda: -round(random.choice([7.99, 12.99, 17.99]), 2)),
        ("SPOTIFY {n}", lambda: -round(random.choice([4.99, 9.99, 14.99]), 2)),
        ("AMAZON PRIME {n}", lambda: -round(random.choice([4.99, 49.99]), 2)),
        ("DISNEY PLUS {n}", lambda: -round(random.choice([5.99, 8.99, 11.99]), 2)),
        ("HBO MAX {n}", lambda: -round(random.choice([4.99, 8.99]), 2)),
        ("ADOBE CREATIVE {n}", lambda: -round(random.choice([12.99, 54.99]), 2)),
        ("APPLE ONE {n}", lambda: -round(random.choice([14.95, 21.95]), 2)),
        ("MICROSOFT 365 {n}", lambda: -round(random.choice([7.0, 10.0, 12.0]), 2)),
    ],
    "transporte": [
        ("EMT MADRID {n}", lambda: -round(random.uniform(1.5, 60), 2)),
        ("RENFE CERCANIAS {n}", lambda: -round(random.uniform(1.7, 40), 2)),
        ("CABIFY {n}", lambda: -round(random.uniform(5, 35), 2)),
        ("UBER {n}", lambda: -round(random.uniform(4, 40), 2)),
        ("GASOLINERA REPSOL {n}", lambda: -round(random.uniform(20, 120), 2)),
        ("PARKING EMPARK {n}", lambda: -round(random.uniform(2, 30), 2)),
        ("BLABLACAR {n}", lambda: -round(random.uniform(5, 40), 2)),
        ("METRO MADRID {n}", lambda: -round(random.uniform(1.5, 30), 2)),
    ],
    "transferencias": [
        ("BIZUM ENVIADO {n}", lambda: -round(random.uniform(5, 500), 2)),
        ("TRANSFERENCIA EMITIDA {n}", lambda: -round(random.uniform(10, 2000), 2)),
        ("BIZUM PAGO {n}", lambda: -round(random.uniform(5, 200), 2)),
        ("TRASPASO CUENTA {n}", lambda: -round(random.uniform(50, 3000), 2)),
        ("PAGO ENTRE CUENTAS {n}", lambda: -round(random.uniform(20, 500), 2)),
    ],
    "otros": [
        ("COMISION BANCARIA {n}", lambda: -round(random.uniform(1, 30), 2)),
        ("CARGO DESCONOCIDO {n}", lambda: -round(random.uniform(1, 50), 2)),
        ("DEVOLUCION CARGO {n}", lambda: round(random.uniform(1, 50), 2)),
        ("CONCEPTO NO IDENTIFICADO {n}", lambda: -round(random.uniform(1, 100), 2)),
    ],
}

TARGET_PER_CLASS = 200

def generate_synthetic(target=TARGET_PER_CLASS):
    rows = []
    for cat, templates in SYNTHETIC_TEMPLATES.items():
        for i in range(target):
            tmpl, amount_fn = random.choice(templates)
            desc = tmpl.replace("{n}", str(random.randint(1, 9999)))
            rows.append({
                "descripcion": desc,
                "importe": amount_fn(),
                "categoria": cat,
                "comercio_normalizado": None,
                "source": "synthetic",
            })
    return pd.DataFrame(rows)

df_synth = generate_synthetic(TARGET_PER_CLASS)
print(f"Datos sintéticos generados: {len(df_synth)} registros")
print(df_synth["categoria"].value_counts().to_string())


### 2.4 Combinar y split 80/10/10 estratificado

In [ ]:
df_all = pd.concat([df_real, df_synth], ignore_index=True)
df_all = df_all.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Total combinado: {len(df_all)} registros")

# Split estratificado
train_val, df_test = train_test_split(
    df_all, test_size=0.10, stratify=df_all["categoria"], random_state=SEED
)
df_train, df_val = train_test_split(
    train_val, test_size=0.111, stratify=train_val["categoria"], random_state=SEED
)
# 0.111 × 0.90 ≈ 0.10 del total → 80/10/10

print(f"\nSplit:")
print(f"  Train : {len(df_train):>4}  ({len(df_train)/len(df_all)*100:.1f}%)")
print(f"  Val   : {len(df_val):>4}  ({len(df_val)/len(df_all)*100:.1f}%)")
print(f"  Test  : {len(df_test):>4}  ({len(df_test)/len(df_all)*100:.1f}%)")

# Visualización de la distribución
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, df_) in zip(axes, [("Train", df_train), ("Val", df_val), ("Test", df_test)]):
    counts = df_["categoria"].value_counts()
    ax.barh(counts.index, counts.values, color="#4C72B0")
    ax.set_title(f"{name} ({len(df_)} muestras)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Nº muestras")
    for i, v in enumerate(counts.values):
        ax.text(v + 0.5, i, str(v), va="center", fontsize=8)
plt.suptitle("Distribución por categoría tras el split", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 3. Embeddings

In [ ]:
# Modelo multilingüe ligero, optimizado para frases cortas en español
EMBEDDER_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

print(f"Cargando embedder: {EMBEDDER_NAME} ...")
embedder = SentenceTransformer(EMBEDDER_NAME)
print(f"Dimensión de embedding: {embedder.get_sentence_embedding_dimension()}")


In [ ]:
def make_features(df: pd.DataFrame) -> np.ndarray:
    """
    Concatena embedding de descripción + importe normalizado.
    El importe añade señal discriminante (p.ej. ingresos > 0, suscripciones < 20).
    """
    descs = df["descripcion"].fillna("").tolist()
    embs  = embedder.encode(descs, batch_size=256, show_progress_bar=True,
                             normalize_embeddings=True)
    # Normalizar el importe al rango [-1, 1] con clip
    amounts = np.clip(df["importe"].fillna(0).values / 3000.0, -1, 1).reshape(-1, 1)
    return np.hstack([embs, amounts])

print("Generando embeddings para train...")
X_train = make_features(df_train)
y_train = df_train["categoria"].values

print("Generando embeddings para val...")
X_val   = make_features(df_val)
y_val   = df_val["categoria"].values

print("Generando embeddings para test...")
X_test  = make_features(df_test)
y_test  = df_test["categoria"].values

print(f"\nShape X_train: {X_train.shape}")
print(f"Shape X_val:   {X_val.shape}")
print(f"Shape X_test:  {X_test.shape}")


## 4. Entrenamiento — comparativa LogReg / LinearSVC / LightGBM

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import lightgbm as lgb

le = LabelEncoder()
le.fit(CATEGORIES)
y_train_enc = le.transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

results = {}

# ── 4.1 Logistic Regression ───────────────────────────────────────────────────
print("Entrenando Logistic Regression...")
t0 = time.time()
lr = LogisticRegression(max_iter=1000, C=4.0, solver="lbfgs",
                        multi_class="multinomial", random_state=SEED, n_jobs=-1)
lr.fit(X_train, y_train_enc)
val_acc_lr = accuracy_score(y_val_enc, lr.predict(X_val))
print(f"  Val accuracy: {val_acc_lr:.4f}  ({time.time()-t0:.1f}s)")
results["LogReg"] = {"model": lr, "val_acc": val_acc_lr}

# ── 4.2 LinearSVC (calibrado) ─────────────────────────────────────────────────
print("Entrenando LinearSVC + calibración...")
t0 = time.time()
svc_base = LinearSVC(C=1.0, max_iter=2000, random_state=SEED)
svc = CalibratedClassifierCV(svc_base, cv=3, method="isotonic")
svc.fit(X_train, y_train_enc)
val_acc_svc = accuracy_score(y_val_enc, svc.predict(X_val))
print(f"  Val accuracy: {val_acc_svc:.4f}  ({time.time()-t0:.1f}s)")
results["LinearSVC"] = {"model": svc, "val_acc": val_acc_svc}

# ── 4.3 LightGBM ─────────────────────────────────────────────────────────────
print("Entrenando LightGBM...")
t0 = time.time()
lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    num_leaves=63,
    learning_rate=0.08,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)
lgbm.fit(X_train, y_train_enc,
         eval_set=[(X_val, y_val_enc)],
         callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)])
val_acc_lgbm = accuracy_score(y_val_enc, lgbm.predict(X_val))
print(f"  Val accuracy: {val_acc_lgbm:.4f}  ({time.time()-t0:.1f}s)")
results["LightGBM"] = {"model": lgbm, "val_acc": val_acc_lgbm}

# ── Comparativa ───────────────────────────────────────────────────────────────
print("\n{'='*45}")
print("COMPARATIVA EN VALIDACIÓN")
print("="*45)
for name, r in sorted(results.items(), key=lambda x: -x[1]["val_acc"]):
    print(f"  {name:<12}  val_acc = {r['val_acc']:.4f}")


### 4.1 Selección del mejor modelo y calibración de probabilidades

In [ ]:
# Seleccionar el mejor modelo de validación
best_name = max(results, key=lambda k: results[k]["val_acc"])
best_model = results[best_name]["model"]
print(f"Mejor modelo: {best_name}  (val_acc={results[best_name]['val_acc']:.4f})")

# Calibrar probabilidades si el modelo no las tiene ya calibradas
if best_name == "LogReg":
    # LogReg ya tiene predict_proba calibrada; hacemos isotonic sobre val para afinar
    clf_calibrated = CalibratedClassifierCV(best_model, cv="prefit", method="isotonic")
    clf_calibrated.fit(X_val, y_val_enc)
    print("Calibración isotónica aplicada sobre val set.")
elif best_name == "LightGBM":
    # LightGBM tiene predict_proba nativa
    clf_calibrated = best_model
    print("LightGBM tiene probabilidades nativas (no requiere calibración extra).")
else:
    # LinearSVC ya fue calibrado durante el entrenamiento
    clf_calibrated = best_model
    print("LinearSVC ya está calibrado.")

# Verificar calibración: distribución de max_proba en val
proba_val = clf_calibrated.predict_proba(X_val)
max_proba  = proba_val.max(axis=1)

plt.figure(figsize=(8, 3))
plt.hist(max_proba, bins=40, color="#4C72B0", edgecolor="white")
plt.xlabel("Probabilidad máxima predicha")
plt.ylabel("Frecuencia")
plt.title(f"Distribución de confianza — {best_name} (val set)")
plt.axvline(0.7, color="red", linestyle="--", label="umbral 0.70")
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nMedia confianza max: {max_proba.mean():.3f}")
print(f"% transacciones con confianza > 0.7: {(max_proba > 0.7).mean()*100:.1f}%")
print(f"% transacciones con confianza < 0.5: {(max_proba < 0.5).mean()*100:.1f}%  (← candidatas a LLM fallback)")


### 4.2 Umbral de confianza

In [ ]:
# Barrido de umbral para encontrar el óptimo accuracy vs % escalado al LLM
thresholds = np.arange(0.30, 0.95, 0.05)
pred_val   = clf_calibrated.predict(X_val)
proba_val  = clf_calibrated.predict_proba(X_val)
max_proba_val = proba_val.max(axis=1)

rows_thresh = []
for thr in thresholds:
    high_conf = max_proba_val >= thr
    if high_conf.sum() == 0:
        continue
    acc_high = accuracy_score(y_val_enc[high_conf], pred_val[high_conf])
    pct_llm  = (~high_conf).mean() * 100
    rows_thresh.append({"umbral": thr, "accuracy_nivel1": acc_high, "pct_llm": pct_llm})

df_thresh = pd.DataFrame(rows_thresh)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.plot(df_thresh["umbral"], df_thresh["accuracy_nivel1"], "b-o", label="Accuracy nivel 1")
ax2.plot(df_thresh["umbral"], df_thresh["pct_llm"], "r--s", label="% escalado LLM")
ax1.set_xlabel("Umbral de confianza")
ax1.set_ylabel("Accuracy (nivel 1)", color="blue")
ax2.set_ylabel("% transacciones → LLM", color="red")
ax1.set_ylim(0.5, 1.0)
ax2.set_ylim(0, 100)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")
plt.title("Accuracy vs % escalado al LLM según umbral de confianza")
plt.tight_layout()
plt.show()

# Elegir umbral: máximo accuracy con < 20% escalado
optimal = df_thresh[df_thresh["pct_llm"] < 20].sort_values("accuracy_nivel1", ascending=False).iloc[0]
CONFIDENCE_THRESHOLD = float(optimal["umbral"])
print(f"\nUmbral seleccionado: {CONFIDENCE_THRESHOLD:.2f}")
print(f"  Accuracy nivel 1 sobre val: {optimal['accuracy_nivel1']:.4f}")
print(f"  % que escala al LLM:        {optimal['pct_llm']:.1f}%")


## 5. Fallback LLM — Nivel 2

In [ ]:
# ── Prompt con few-shot examples para clasificación de baja confianza ─────────

FEW_SHOT_EXAMPLES = [
    {"desc": "MERCADONA 1234", "amount": -45.20, "categoria": "alimentacion"},
    {"desc": "NETFLIX SUSCRIPCION", "amount": -12.99, "categoria": "suscripciones"},
    {"desc": "EMT MADRID ABONO", "amount": -20.0, "categoria": "transporte"},
    {"desc": "NOMINA ENERO", "amount": 2100.0, "categoria": "ingresos"},
    {"desc": "FARMACIA CENTRAL", "amount": -18.50, "categoria": "salud"},
    {"desc": "RESTAURANTE EL RINCON", "amount": -32.0, "categoria": "restauracion"},
    {"desc": "AMAZON COMPRA", "amount": -67.0, "categoria": "compras"},
    {"desc": "BIZUM ENVIADO", "amount": -50.0, "categoria": "transferencias"},
    {"desc": "AGENCIA TRIBUTARIA", "amount": -320.0, "categoria": "impuestos_tasas"},
    {"desc": "IKEA ONLINE", "amount": -189.0, "categoria": "hogar"},
    {"desc": "STEAM JUEGO", "amount": -19.99, "categoria": "ocio"},
    {"desc": "CARGO DESCONOCIDO", "amount": -3.5, "categoria": "otros"},
]

CATEGORIES_STR = ", ".join(CATEGORIES)
FEW_SHOT_STR = "\n".join(
    f'  Descripción: "{e["desc"]}", Importe: {e["amount"]} → {e["categoria"]}'
    for e in FEW_SHOT_EXAMPLES
)

def build_llm_prompt(description: str, amount: float) -> str:
    return f"""Eres un clasificador de transacciones bancarias en español.
Debes categorizar la transacción en EXACTAMENTE UNA de estas categorías:
{CATEGORIES_STR}

Ejemplos:
{FEW_SHOT_STR}

Ahora clasifica esta transacción:
  Descripción: "{description}", Importe: {amount}

Responde ÚNICAMENTE con el nombre de la categoría, sin explicación ni puntuación."""


# ── Cliente LLM (Azure OpenAI o Anthropic según disponibilidad) ────────────────

import os

def call_llm_level2(description: str, amount: float) -> str:
    """
    Llama al LLM para clasificar una transacción de baja confianza.
    Prioridad: Azure OpenAI → OpenAI → mock
    Devuelve la categoría como string.
    """
    prompt = build_llm_prompt(description, amount)

    # Intentar Azure OpenAI
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    azure_key      = os.getenv("AZURE_OPENAI_KEY")
    azure_deploy   = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

    if azure_endpoint and azure_key:
        try:
            import openai
            client = openai.AzureOpenAI(
                azure_endpoint=azure_endpoint,
                api_key=azure_key,
                api_version="2024-02-01",
            )
            resp = client.chat.completions.create(
                model=azure_deploy,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=20,
                temperature=0.0,
            )
            cat = resp.choices[0].message.content.strip().lower()
            return cat if cat in CATEGORIES else "otros"
        except Exception as e:
            print(f"  [LLM] Azure error: {e}")

    # Mock de fallback para desarrollo/tests
    # En producción esto nunca debería ejecutarse
    print(f"  [LLM] Mock — sin credenciales Azure. Transacción: '{description}'")
    return "otros"


# Test del prompt
print("Ejemplo de prompt nivel 2:")
print("─" * 60)
print(build_llm_prompt("PAGO GOOGLE STORAGE 2TB", -2.99))


In [ ]:
# Medir % de transacciones que escalan al LLM en el test set
proba_test    = clf_calibrated.predict_proba(X_test)
max_proba_test = proba_test.max(axis=1)
pct_llm_test  = (max_proba_test < CONFIDENCE_THRESHOLD).mean() * 100

print(f"Test set: {pct_llm_test:.1f}% de transacciones escalan al LLM (umbral={CONFIDENCE_THRESHOLD:.2f})")
print(f"Coste estimado si LLM = €0.0002/llamada y hay 10.000 tx/mes:")
n_llm = 10000 * (pct_llm_test / 100)
print(f"  → {n_llm:.0f} llamadas LLM/mes ≈ €{n_llm * 0.0002:.2f}/mes")


## 6. Anonimización pre-LLM (Presidio + regex)

In [ ]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# ── Construir el analizador con recognizers personalizados ────────────────────

def build_analyzer():
    analyzer = AnalyzerEngine()

    # IBAN
    analyzer.registry.add_recognizer(PatternRecognizer(
        supported_entity="IBAN",
        patterns=[Pattern("IBAN", r"\b[A-Z]{2}\d{2}[\s\-]?(?:\d{4}[\s\-]?){4}\d{0,4}\b", 0.95)],
        supported_language="es"
    ))
    # DNI / NIE
    analyzer.registry.add_recognizer(PatternRecognizer(
        supported_entity="DNI",
        patterns=[Pattern("DNI", r"\b\d{7,8}[A-Z]\b|\b[XYZ]\d{7}[A-Z]\b", 0.90)],
        supported_language="es"
    ))
    # Número de cuenta CCC
    analyzer.registry.add_recognizer(PatternRecognizer(
        supported_entity="CUENTA_BANCARIA",
        patterns=[Pattern("CCC", r"\b\d{4}[\s\-]\d{4}[\s\-]\d{2}[\s\-]\d{10}\b|\b\d{20}\b", 0.85)],
        supported_language="es"
    ))
    # Teléfono español
    analyzer.registry.add_recognizer(PatternRecognizer(
        supported_entity="TELEFONO",
        patterns=[Pattern("TEL", r"\b[679]\d{8}\b", 0.75)],
        supported_language="es"
    ))
    return analyzer

BIZUM_NAME_RE = re.compile(
    r"(bizum\s+(?:de\s+|para\s+|a\s+)?)([A-ZÁÉÍÓÚÑÜ][a-záéíóúñü]+(?:\s+[A-ZÁÉÍÓÚÑÜ][a-záéíóúñü]+){0,3})",
    re.IGNORECASE
)

_analyzer   = build_analyzer()
_anonymizer = AnonymizerEngine()
ENTITY_TYPES = ["IBAN", "DNI", "CUENTA_BANCARIA", "TELEFONO", "PERSON", "EMAIL_ADDRESS"]

def anonymize(text: str):
    if not text or not text.strip():
        return text, []
    entities_found = []

    # 1. Regex para nombres en Bizums
    def replace_bizum_name(m):
        entities_found.append({"type": "PERSON", "text": m.group(2), "source": "regex"})
        return m.group(1) + "<PERSON>"
    text_clean = BIZUM_NAME_RE.sub(replace_bizum_name, text)

    # 2. Presidio para el resto
    results = _analyzer.analyze(text=text_clean, entities=ENTITY_TYPES, language="es")
    for r in results:
        entities_found.append({"type": r.entity_type, "text": text_clean[r.start:r.end], "score": r.score})
    if results:
        anon_result = _anonymizer.anonymize(
            text=text_clean,
            analyzer_results=results,
            operators={et: OperatorConfig("replace", {"new_value": f"<{et}>"}) for et in ENTITY_TYPES},
        )
        text_clean = anon_result.text
    return text_clean, entities_found

print("Analizador Presidio construido ✓")


In [ ]:
# ── Validación del anonimizador ──────────────────────────────────────────────

test_cases = [
    ("BIZUM DE Juan García Martínez restaurante cena", True),
    ("TRANSFERENCIA ES91 2100 0418 42 0200051332", True),
    ("PAGO DNI 12345678Z MULTA DGT", True),
    ("CONTACTO 612345678 SERVICIO TECNICO", True),
    ("NOMINA EMPRESA usuario@empresa.com", True),
    ("MERCADONA COMPRA ALIMENTACION", False),
    ("NETFLIX SUSCRIPCION MENSUAL", False),
    ("UBER VIAJE 13.50EUR", False),
]

print(f"{'Texto original':<50} {'Con PII':^8} {'Detectado':^9} {'Texto anonimizado'}")
print("─" * 120)
tp = fp = fn = tn = 0
for text, has_pii in test_cases:
    anon_text, ents = anonymize(text)
    detected = len(ents) > 0
    if   detected and has_pii:  tp += 1; status = "✓"
    elif detected and not has_pii: fp += 1; status = "⚠️ FP"
    elif not detected and has_pii: fn += 1; status = "✗ FN"
    else: tn += 1; status = "✓"
    print(f"{status} {text:<48} {str(has_pii):^8} {str(detected):^9} {anon_text}")

total_pii = sum(1 for _, h in test_cases if h)
print(f"\nRecall PII:   {tp}/{total_pii} = {tp/total_pii:.0%}")
print(f"Falsos pos:   {fp}")
print(f"Falsos neg:   {fn}")
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_anon   = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
print(f"F1 anonimizador: {f1_anon:.2f}")


## 7. Detección de anomalías

In [ ]:
# ── 7.1 Perfil de usuario y Z-score por categoría ────────────────────────────

def build_user_profile(df: pd.DataFrame) -> dict:
    """Calcula media y std de importe por categoría (solo negativos = gastos)."""
    gastos = df[df["importe"] < 0].copy()
    gastos["importe_abs"] = gastos["importe"].abs()
    profile = {}
    for cat, grp in gastos.groupby("categoria"):
        profile[cat] = {
            "mean": grp["importe_abs"].mean(),
            "std":  grp["importe_abs"].std() or 1.0,
            "n":    len(grp),
        }
    return profile

def compute_zscore(amount: float, categoria: str, profile: dict) -> float:
    """Z-score del importe absoluto dentro de su categoría."""
    if categoria not in profile or amount >= 0:
        return 0.0
    p = profile[categoria]
    return (abs(amount) - p["mean"]) / p["std"]

ZSCORE_THRESHOLD = 3.0  # Flag si Z > 3

# Construir perfil sobre train
profile = build_user_profile(df_train)

print("Perfil de usuario (media ± std de gasto por categoría):")
df_profile = pd.DataFrame(profile).T.sort_values("mean", ascending=False)
print(df_profile[["mean", "std", "n"]].round(2).to_string())


In [ ]:
# ── 7.2 Inyectar outliers y medir precision/recall ───────────────────────────

def inject_outliers(df: pd.DataFrame, n: int = 100) -> pd.DataFrame:
    """Inyecta transacciones outlier multiplicando el importe × [5, 20]."""
    outliers = df.sample(n, random_state=SEED).copy()
    outliers["importe"] = outliers["importe"].apply(
        lambda x: x * random.uniform(5, 20) if x < 0 else x
    )
    outliers["is_anomaly"] = True
    return outliers

df_test_eval = df_test.copy()
df_test_eval["is_anomaly"] = False

df_outliers   = inject_outliers(df_test_eval, n=100)
df_test_final = pd.concat([df_test_eval, df_outliers], ignore_index=True)

# Detectar anomalías
df_test_final["zscore"] = df_test_final.apply(
    lambda r: compute_zscore(r["importe"], r["categoria"], profile), axis=1
)
df_test_final["flag_anomaly"] = df_test_final["zscore"] > ZSCORE_THRESHOLD

# Métricas
from sklearn.metrics import classification_report as cr
y_true_anom = df_test_final["is_anomaly"].astype(int)
y_pred_anom = df_test_final["flag_anomaly"].astype(int)
print("Detección de anomalías (Z-score):")
print(cr(y_true_anom, y_pred_anom, target_names=["normal", "anomalía"]))

# Distribución de Z-scores
fig, ax = plt.subplots(figsize=(9, 3))
normal  = df_test_final[~df_test_final["is_anomaly"]]["zscore"]
anomaly = df_test_final[ df_test_final["is_anomaly"]]["zscore"]
ax.hist(normal, bins=60, alpha=0.6, label="Normal", color="#4C72B0")
ax.hist(anomaly, bins=30, alpha=0.7, label="Outlier inyectado", color="#DD4444")
ax.axvline(ZSCORE_THRESHOLD, color="red", linestyle="--", label=f"Umbral Z={ZSCORE_THRESHOLD}")
ax.set_xlabel("Z-score")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de Z-scores: normales vs outliers")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── 7.3 Detector de suscripciones recurrentes ────────────────────────────────

def detect_subscriptions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrupa por (comercio_normalizado, importe_redondeado).
    Detecta periodicidad y flag si el importe cambia.
    """
    df_subs = df[df["categoria"] == "suscripciones"].copy()
    df_subs["importe_abs"] = df_subs["importe"].abs().round(2)

    # Añadir fecha sintética para demo (producción usaría la fecha real)
    np.random.seed(SEED)
    df_subs["mes"] = np.random.randint(1, 13, size=len(df_subs))

    grouped = (
        df_subs.groupby(["comercio_normalizado", "importe_abs"])
        .agg(apariciones=("mes", "count"), meses=("mes", list))
        .reset_index()
        .sort_values("apariciones", ascending=False)
    )
    grouped["periodicidad"] = grouped["apariciones"].apply(
        lambda n: "mensual" if n >= 3 else ("trimestral" if n >= 2 else "puntual")
    )
    print(f"Suscripciones detectadas: {len(grouped)}")
    print(grouped[["comercio_normalizado", "importe_abs", "apariciones", "periodicidad"]].head(15).to_string(index=False))
    return grouped

subs_detected = detect_subscriptions(df_train)


## 8. Insights mensuales — pipeline batch → LLM

In [ ]:
def compute_monthly_aggregates(df: pd.DataFrame) -> dict:
    """
    Calcula agregados para el prompt de insights:
    top categorías de gasto, variaciones, suscripciones.
    """
    gastos = df[df["importe"] < 0].copy()
    gastos["importe_abs"] = gastos["importe"].abs()

    top_cats = (
        gastos.groupby("categoria")["importe_abs"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
    )
    total = gastos["importe_abs"].sum()
    n_subs = len(gastos[gastos["categoria"] == "suscripciones"])

    return {
        "total_gastado": round(total, 2),
        "top_categorias": {k: round(v, 2) for k, v in top_cats.items()},
        "n_suscripciones": n_subs,
        "n_transacciones": len(df),
    }

def build_insights_prompt(agg: dict, mes: str = "Mayo 2025") -> str:
    top_str = "\n".join(f"  - {k}: {v}€" for k, v in agg["top_categorias"].items())
    return f"""Eres un asistente financiero personal. Redacta un resumen mensual
breve (3-4 frases) y accionable en español para el usuario, basándote SOLO en estos datos:

Mes: {mes}
Total gastado: {agg["total_gastado"]}€
Número de transacciones: {agg["n_transacciones"]}
Top categorías de gasto:
{top_str}
Suscripciones activas detectadas: {agg["n_suscripciones"]}

Formato de respuesta:
- Frase 1: resumen general del mes
- Frase 2: categoría de mayor gasto + insight accionable
- Frase 3: aviso sobre suscripciones si hay muchas (>5)
- Frase 4: consejo de ahorro personalizado"""

# Demo
agg = compute_monthly_aggregates(df_train)
print("=== AGREGADOS ===")
for k, v in agg.items():
    print(f"  {k}: {v}")

print("\n=== PROMPT PARA EL LLM ===")
print(build_insights_prompt(agg))


## 9. Evaluación final sobre test set

In [ ]:
# ── Predicciones nivel 1 ──────────────────────────────────────────────────────
proba_test  = clf_calibrated.predict_proba(X_test)
pred_test   = clf_calibrated.predict(X_test)
max_conf    = proba_test.max(axis=1)

# Accuracy global nivel 1
acc_l1 = accuracy_score(y_test_enc, pred_test)
f1_l1  = f1_score(y_test_enc, pred_test, average="macro")
print(f"=== NIVEL 1 ({best_name}) ===")
print(f"Accuracy: {acc_l1:.4f}")
print(f"F1 macro: {f1_l1:.4f}")
print()
print(classification_report(y_test_enc, pred_test, target_names=le.classes_))


In [ ]:
# ── Nivel 1 + 2: simular LLM perfecto en casos de baja confianza ────────────
# (En producción, call_llm_level2 haría la llamada real)
# Para la evaluación académica: los casos <umbral se resuelven con la etiqueta real

low_conf_mask = max_conf < CONFIDENCE_THRESHOLD
pred_combined = pred_test.copy()
# Simular LLM perfecto
pred_combined[low_conf_mask] = y_test_enc[low_conf_mask]

acc_l2 = accuracy_score(y_test_enc, pred_combined)
f1_l2  = f1_score(y_test_enc, pred_combined, average="macro")

print(f"=== NIVEL 1 + 2 (LLM perfecto en baja confianza) ===")
print(f"Transacciones escaladas al LLM: {low_conf_mask.sum()} ({low_conf_mask.mean()*100:.1f}%)")
print(f"Accuracy: {acc_l2:.4f}  (mejora: +{acc_l2-acc_l1:.4f})")
print(f"F1 macro: {f1_l2:.4f}  (mejora: +{f1_l2-f1_l1:.4f})")

# Comparativa visual
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(
    [f"Solo nivel 1\n({best_name})", "Nivel 1 + 2\n(LLM fallback)"],
    [acc_l1, acc_l2],
    color=["#4C72B0", "#55A868"],
    width=0.5
)
for bar, val in zip(bars, [acc_l1, acc_l2]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.002, f"{val:.4f}",
            ha="center", fontsize=11, fontweight="bold")
ax.set_ylim(min(acc_l1, acc_l2) - 0.05, 1.0)
ax.set_ylabel("Accuracy")
ax.set_title("Comparativa: solo nivel 1 vs nivel 1 + 2")
plt.tight_layout()
plt.show()


In [ ]:
# ── Matriz de confusión ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test_enc, pred_test)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Absoluta
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=le.classes_, yticklabels=le.classes_)
axes[0].set_title(f"Matriz de confusión — {best_name} (absoluta)", fontsize=12)
axes[0].set_xlabel("Predicho"); axes[0].set_ylabel("Real")
axes[0].tick_params(axis="x", rotation=45)

# Porcentual
sns.heatmap(cm_pct, annot=True, fmt=".2f", cmap="Blues", ax=axes[1],
            xticklabels=le.classes_, yticklabels=le.classes_, vmin=0, vmax=1)
axes[1].set_title(f"Matriz de confusión — {best_name} (proporción por fila)", fontsize=12)
axes[1].set_xlabel("Predicho"); axes[1].set_ylabel("Real")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# ── Análisis de errores ──────────────────────────────────────────────────────
df_test_errors = df_test.copy().reset_index(drop=True)
df_test_errors["pred"] = le.inverse_transform(pred_test)
df_test_errors["real"] = y_test
df_test_errors["confianza"] = max_conf
df_test_errors["correcto"] = df_test_errors["pred"] == df_test_errors["real"]

errores = df_test_errors[~df_test_errors["correcto"]]
print(f"Total errores: {len(errores)} sobre {len(df_test)} test ({len(errores)/len(df_test)*100:.1f}%)")

print("\nTop 10 confusiones más frecuentes:")
confusion_pairs = errores.groupby(["real", "pred"]).size().reset_index(name="n").sort_values("n", ascending=False)
print(confusion_pairs.head(10).to_string(index=False))

print("\nEjemplos de errores de alta confianza (modelo seguro pero incorrecto):")
high_conf_errors = errores[errores["confianza"] > 0.8].sort_values("confianza", ascending=False)
print(high_conf_errors[["descripcion", "importe", "real", "pred", "confianza"]].head(10).to_string(index=False))


In [ ]:
# ── F1 por categoría ────────────────────────────────────────────────────────
f1_per_cat = f1_score(y_test_enc, pred_test, average=None)
df_f1 = pd.DataFrame({"categoria": le.classes_, "f1": f1_per_cat}).sort_values("f1")

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#DD4444" if f < 0.7 else "#4C72B0" for f in df_f1["f1"]]
ax.barh(df_f1["categoria"], df_f1["f1"], color=colors)
ax.axvline(0.7, color="red", linestyle="--", alpha=0.5, label="Mínimo aceptable (0.70)")
ax.set_xlabel("F1-score")
ax.set_title("F1 por categoría (test set)")
ax.set_xlim(0, 1.05)
for i, (cat, f) in enumerate(zip(df_f1["categoria"], df_f1["f1"])):
    ax.text(f + 0.01, i, f"{f:.3f}", va="center", fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()


## 10. Serialización de artefactos — `/models`

In [ ]:
import joblib, json
from pathlib import Path
from datetime import datetime

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# ── Guardar el clasificador calibrado ────────────────────────────────────────
classifier_path = MODELS_DIR / "classifier.pkl"
joblib.dump({
    "model": clf_calibrated,
    "label_encoder": le,
    "threshold": CONFIDENCE_THRESHOLD,
    "embedder_name": EMBEDDER_NAME,
}, classifier_path)
print(f"✓ Clasificador guardado: {classifier_path}")

# ── Metadata ─────────────────────────────────────────────────────────────────
metadata = {
    "model_name": best_name,
    "embedder": EMBEDDER_NAME,
    "embedding_dim": embedder.get_sentence_embedding_dimension(),
    "trained_on": datetime.now().isoformat(),
    "categories": CATEGORIES,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "metrics": {
        "accuracy_val": float(results[best_name]["val_acc"]),
        "accuracy_test_l1": float(acc_l1),
        "accuracy_test_l1_l2": float(acc_l2),
        "f1_macro_test": float(f1_l1),
        "pct_llm_fallback": float(pct_llm_test),
    },
    "n_train": len(df_train),
    "n_val": len(df_val),
    "n_test": len(df_test),
}

metadata_path = MODELS_DIR / "model_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"✓ Metadata guardada: {metadata_path}")
print(json.dumps(metadata, indent=2, ensure_ascii=False))


In [ ]:
# ── Medir latencia de inferencia ─────────────────────────────────────────────

N_BENCH = 200
test_samples = df_test.sample(N_BENCH, random_state=SEED).reset_index(drop=True)

times = []
for _, row in test_samples.iterrows():
    t0 = time.perf_counter()
    desc = str(row["descripcion"])
    amt  = float(row["importe"])
    emb  = embedder.encode([desc], normalize_embeddings=True)
    feat = np.hstack([emb, np.clip([[amt / 3000.0]], -1, 1)])
    proba = clf_calibrated.predict_proba(feat)[0]
    pred  = clf_calibrated.predict(feat)[0]
    conf  = proba.max()
    times.append(time.perf_counter() - t0)

p50 = np.percentile(times, 50) * 1000
p95 = np.percentile(times, 95) * 1000
p99 = np.percentile(times, 99) * 1000
mean_ms = np.mean(times) * 1000

print(f"Latencia de inferencia (nivel 1) sobre {N_BENCH} muestras:")
print(f"  Media:  {mean_ms:.2f} ms")
print(f"  p50:    {p50:.2f} ms")
print(f"  p95:    {p95:.2f} ms")
print(f"  p99:    {p99:.2f} ms")

# Guardar en metadata
metadata["latency_ms"] = {"mean": round(mean_ms, 3), "p50": round(p50, 3),
                           "p95": round(p95, 3), "p99": round(p99, 3)}
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("\n✓ Latencia añadida a model_metadata.json")


In [ ]:

# Write load_classifier.py using list of strings to avoid triple-quote nesting
load_lines = [
    '"""\n',
    "load_classifier.py\n",
    "Importar y usar:\n",
    "    from load_classifier import load\n",
    "    classify = load()\n",
    '    result = classify("MERCADONA COMPRA", -45.20)\n',
    '"""\n',
    "import json, time\n",
    "from pathlib import Path\n",
    "import numpy as np\n",
    "import joblib\n",
    "from sentence_transformers import SentenceTransformer\n",
    "\n",
    "MODELS_DIR = Path(__file__).parent\n",
    "_CACHE: dict = {}\n",
    "\n",
    "CATEGORIES = [\n",
    '    "alimentacion", "compras", "hogar", "impuestos_tasas",\n',
    '    "ingresos", "ocio", "otros", "restauracion",\n',
    '    "salud", "suscripciones", "transporte", "transferencias",\n',
    "]\n",
    "\n",
    "def load():\n",
    '    """Carga el clasificador (singleton) y devuelve classify."""\n',
    '    if "classify" in _CACHE:\n',
    '        return _CACHE["classify"]\n',
    '    artifact  = joblib.load(MODELS_DIR / "classifier.pkl")\n',
    '    model     = artifact["model"]\n',
    '    le        = artifact["label_encoder"]\n',
    '    threshold = artifact["threshold"]\n',
    '    embedder  = SentenceTransformer(artifact["embedder_name"])\n',
    "\n",
    "    def classify(description: str, amount: float) -> dict:\n",
    '        """\n',
    "        Clasifica una transaccion bancaria anonimizada.\n",
    "        Returns: {categoria, confianza, nivel_usado}\n",
    '        """\n',
    "        emb  = embedder.encode([description], normalize_embeddings=True)\n",
    "        feat = np.hstack([emb, np.clip([[amount / 3000.0]], -1, 1)])\n",
    "        proba    = model.predict_proba(feat)[0]\n",
    "        pred_idx = int(np.argmax(proba))\n",
    "        conf     = float(proba[pred_idx])\n",
    "        cat      = le.inverse_transform([pred_idx])[0]\n",
    "        if conf >= threshold:\n",
    '            return {"categoria": cat, "confianza": round(conf, 4), "nivel_usado": 1}\n',
    "        try:\n",
    "            cat_llm = _call_llm(description, amount)\n",
    '            return {"categoria": cat_llm, "confianza": 0.0, "nivel_usado": 2}\n',
    "        except Exception:\n",
    '            return {"categoria": cat, "confianza": round(conf, 4), "nivel_usado": 1}\n',
    "\n",
    '    _CACHE["classify"] = classify\n',
    "    return classify\n",
    "\n",
    "\n",
    "def _call_llm(description: str, amount: float) -> str:\n",
    "    import os\n",
    '    cats = ", ".join(CATEGORIES)\n',
    '    prompt = (f"Clasifica esta transaccion en UNA categoria: {cats}\\n"\n',
    '              f\'Descripcion: "{description}", Importe: {amount}\\n\'\n',
    '              "Responde SOLO con el nombre de la categoria.")\n',
    '    ep  = os.getenv("AZURE_OPENAI_ENDPOINT")\n',
    '    key = os.getenv("AZURE_OPENAI_KEY")\n',
    '    dep = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")\n',
    "    if ep and key:\n",
    "        import openai\n",
    "        client = openai.AzureOpenAI(azure_endpoint=ep, api_key=key, api_version='2024-02-01')\n",
    "        resp = client.chat.completions.create(\n",
    "            model=dep,\n",
    '            messages=[{"role": "user", "content": prompt}],\n',
    "            max_tokens=20, temperature=0.0,\n",
    "        )\n",
    "        cat = resp.choices[0].message.content.strip().lower()\n",
    "        return cat if cat in CATEGORIES else 'otros'\n",
    "    return 'otros'\n",
    "\n",
    "\n",
    'if __name__ == "__main__":\n',
    "    classify = load()\n",
    "    for desc, amt in [\n",
    '        ("MERCADONA 1234", -45.20), ("NETFLIX SUSCRIPCION", -12.99),\n',
    '        ("NOMINA ENERO", 2100.0),  ("FARMACIA CENTRAL", -18.50),\n',
    "    ]:\n",
    "        t0 = time.perf_counter()\n",
    "        r  = classify(desc, amt)\n",
    "        ms = (time.perf_counter() - t0) * 1000\n",
    '        print(f"{desc:<30} {amt:>8.2f}  {r}  ({ms:.1f} ms)")\n',
]

load_path = MODELS_DIR / "load_classifier.py"
with open(load_path, "w") as f:
    f.writelines(load_lines)
print(f"load_classifier.py guardado: {load_path}")



In [ ]:
# ── Test final de classify() ─────────────────────────────────────────────────

print("=== Test de classify() con la API del endpoint ===\n")

# Reimplementar aquí la función para no depender del fichero generado
_loaded_artifact = joblib.load(MODELS_DIR / "classifier.pkl")
_model    = _loaded_artifact["model"]
_le       = _loaded_artifact["label_encoder"]
_threshold = _loaded_artifact["threshold"]

def classify(description: str, amount: float) -> dict:
    emb  = embedder.encode([description], normalize_embeddings=True)
    feat = np.hstack([emb, np.clip([[amount / 3000.0]], -1, 1)])
    proba     = _model.predict_proba(feat)[0]
    pred_idx  = int(np.argmax(proba))
    confianza = float(proba[pred_idx])
    categoria = _le.inverse_transform([pred_idx])[0]
    if confianza >= _threshold:
        return {"categoria": categoria, "confianza": round(confianza, 4), "nivel_usado": 1}
    # Fallback mock (sin credenciales Azure en este entorno)
    return {"categoria": categoria, "confianza": round(confianza, 4), "nivel_usado": 2}

test_inputs = [
    ("MERCADONA 1234",         -45.20),
    ("NETFLIX SUSCRIPCION",    -12.99),
    ("NOMINA ENERO",          2100.00),
    ("FARMACIA CENTRAL",       -18.50),
    ("AMAZON PRIME ANUALPAGO", -49.99),
    ("EMT MADRID ABONO",       -20.00),
    ("AGENCIA TRIBUTARIA",    -450.00),
    ("BIZUM ENVIADO AMIGO",    -50.00),
    ("RESTAURANTE LA TABERNA", -32.00),
    ("CARGO DESCONOCIDO",       -5.00),
]

print(f"{'Descripción':<35} {'Importe':>8}  {'categoria':<18} {'confianza':>10}  nivel")
print("─" * 85)
for desc, amt in test_inputs:
    r = classify(desc, amt)
    print(f"{desc:<35} {amt:>8.2f}  {r['categoria']:<18} {r['confianza']:>10.4f}  L{r['nivel_usado']}")


In [ ]:
# ── Resumen final ────────────────────────────────────────────────────────────
print("=" * 60)
print("RESUMEN DEL MODELO")
print("=" * 60)
print(f"  Modelo base:              {best_name}")
print(f"  Embedder:                 {EMBEDDER_NAME}")
print(f"  Categorías:               {len(CATEGORIES)}")
print(f"  Train / Val / Test:       {len(df_train)} / {len(df_val)} / {len(df_test)}")
print(f"  Accuracy test (nivel 1):  {acc_l1:.4f}")
print(f"  Accuracy test (L1+L2):    {acc_l2:.4f}")
print(f"  F1 macro test:            {f1_l1:.4f}")
print(f"  Umbral confianza:         {CONFIDENCE_THRESHOLD:.2f}")
print(f"  % escalado LLM (test):    {pct_llm_test:.1f}%")
print(f"  Latencia media (L1):      {mean_ms:.2f} ms")
print(f"\nArtefactos en ./models/:")
for p in sorted(MODELS_DIR.iterdir()):
    print(f"  📄 {p.name}  ({p.stat().st_size / 1024:.0f} KB)")
